This notebook is for building our own lightweight version of Claimify. Claimify uses a 4-stage pipeline:
1. Sentence Splitting: Breaks text into individual sentences with surrounding context
2. Selection: Filters for sentences containing verifiable propositions, excluding opinions and speculation
3. Disambiguation: Resolves ambiguities or discards sentences that cannot be clarified
4. Decomposition: Breaks down sentences into atomic, self-contained factual claims

In [21]:
import nltk
from transformers import pipeline
from fastcoref.modeling import FCoref, FCorefModel
import torch

In [22]:
# Download the sentence tokenizer for Stage 1
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/frankiepike/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [23]:
#Initialize the fastcoref model

if not hasattr(FCorefModel, "all_tied_weights_keys"):
    FCorefModel.all_tied_weights_keys = {}

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
coref_model = FCoref(device=device)

04/06/2026 11:27:13 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:27:13 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/06/2026 11:27:13 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:27:13 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
04/06/2026 11:27:13 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/06/2026 11:27:13 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/tokenizer_config.json

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
04/06/2026 11:27:14 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref "HTTP/1.1 200 OK"
04/06/2026 11:27:14 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/main "HTTP/1.1 200 OK"
04/06/2026 11:27:14 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"
04/06/2026 11:27:14 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04/06/2026 11:27:14 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Fou

In [ ]:
# Initialize a lightweight zero-shot classification model for Stage 2
# This model runs locally and requires no API keys.
claim_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3", # Lightweight distilled model
    device=-1 # Set to 0 if you are using a GPU
)

In [24]:
def resolve_coreferences(text):
    """
    Replaces pronouns and implicit references in the text with their explicit entities.
    """
    #Predict coreference clusters
    preds = coref_model.predict(texts=[text])

    #Get clusters as character start/end indices
    clusters = preds[0].get_clusters(as_strings=False)

    replacements = []
    for cluster in clusters:
        #The first mention in a cluster is usually the explicit entity (the antecedent)
        primary_start, primary_end = cluster[0]
        primary_text = text[primary_start:primary_end]

        # We want to replace all subsequent mentions (usually pronouns) with the primary text
        for mention_start, mention_end in cluster[1:]:
            replacements.append((mention_start, mention_end, primary_text))

    #Sort replacements in reverse order of their start index
    replacements.sort(key=lambda x: x[0], reverse=True)

    #Apply replacements
    resolved_text = text
    for start, end, rep_text in replacements:
        resolved_text = resolved_text[:start] + rep_text + resolved_text[end:]

    return resolved_text

In [ ]:
def lightweight_claimify(text, threshold=0.6):
    """
    A keyless approximation of the Claimify pipeline.
    Filters out opinions and returns only sentences likely to be verifiable claims.
    """
    #Stage 1: Sentence Splitting
    sentences = nltk.sent_tokenize(text)
    extracted_claims = []

    #Stage 2: Selection (Filtering opinions vs. verifiable claims)
    for sent in sentences:
        # Ignore extremely short strings
        if len(sent.split()) < 4:
            continue

        result = claim_classifier(
            sent,
            candidate_labels=["factual claim", "personal opinion"],
            multi_label=False
        )

        top_label = result['labels'][0]
        confidence = result['scores'][0]

        #Keep it if the model is confident it's a factual statement rather than an opinion
        if top_label == "factual claim" and confidence >= threshold:
            extracted_claims.append(sent)

    return extracted_claims

In [ ]:
#Testing
sample_article = """
I think the new policies are an absolute disaster.
The inflation rate in the country rose by 4.2% last quarter.
It is the worst economic decision in history!
According to the report, the city council voted 5-2 to pass the infrastructure bill.
This bill will go into effect next year.
"""

In [ ]:
print("Original Text:")
print(sample_article.strip())
print("\n--- Extracted Verifiable Claims ---")
claims = lightweight_claimify(sample_article)
for claim in claims:
    print(f"- {claim}")